# 01 — Predict by the neighbourhood vote

*Module 01 · k-Nearest Neighbours — notebook 1 of 6*

Welcome to your first real machine-learning method. Everything you built in module 00 — features
and a feature space, Euclidean distance, the sealed train/test split — comes together here into a
classifier you can write by hand in a few lines.

The idea is wonderfully direct: **to label a new point, look at the training points nearest to it
and let them vote.** That is k-nearest neighbours (k-NN), and by the end of this notebook you will
have run it yourself, one point at a time.

**Prerequisites:** notebook 02 (features, the feature space, Euclidean distance) and notebook 04
(the train/test split).

**What you will be able to do**

- State the k-NN rule in one sentence.
- Compute it **by hand**: distances from a query to every training point, then the k nearest, then
  the majority vote.
- See **k** for what it is — the size of the neighbourhood that gets to vote.
- Explain why k-NN is called a **lazy** learner, and feel where its cost actually lives.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

from ml_course import viz
from ml_course.colors import CLASS_CYCLE, COLORS

# Fix the seed so every run of this notebook tells the same story.
np.random.seed(0)
viz.use_course_style()

# A 2-D dataset with genuine class overlap (the next section explains why).
X, y = make_moons(n_samples=300, noise=0.30, random_state=0)

# The sealed train/test split from notebook 04: stratified, so both classes keep
# their proportions; random_state fixes which points land where.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=0
)

print(f"training points: {X_train.shape[0]}   test points: {X_test.shape[0]}")
print(f"training class balance: {np.bincount(y_train).tolist()}  (class 0, class 1)")

## A recap, and a new playground

Two tools from module 00 carry us through this whole chapter:

- **Euclidean distance** (notebook 02) — the straight-line distance between two points in the
  feature space. It is how we will measure "nearest".
- **The train/test split** (notebook 04) — we learn from the training points only and keep the test
  points sealed, so we can later ask an honest question: does the rule generalize?

What is new is the **method** itself — and a new dataset to show it on. Until now we used the Palmer
penguins, where the two species separate so cleanly that almost any rule gets them right. That is
comforting, but it hides a method's character. So we switch to `make_moons`: two interleaving, noisy
crescents that genuinely overlap. (This is the same honest switch the course first made back in
module 00, when a clean dataset stopped being instructive.) On data like this, *who your neighbours
are* truly matters — which is exactly the question k-NN answers.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for cls in (0, 1):
    mask = y_train == cls
    ax.scatter(
        X_train[mask, 0],
        X_train[mask, 1],
        color=CLASS_CYCLE[cls],
        edgecolor=COLORS["text"],
        linewidth=0.5,
        s=45,
        alpha=0.9,
        label=f"class {cls}",
    )
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.set_title("The training set: two interleaving, noisy crescents")
ax.legend(loc="upper right")
plt.show()

**Read the figure.** Each point is one training example, coloured by its class. The two crescents
interleave, and through the middle they overlap: there is a band where class-0 and class-1 points
sit side by side. Out in the wings, a point is surrounded by its own class; near that central band,
a point's neighbours are a mix of both. That is where the question "who are your neighbours?" stops
being a formality and starts deciding the answer.

## The idea: k-nearest neighbours

Here is the entire method in one sentence:

> To classify a new point, find the **k** training points closest to it, and predict the **majority
> vote** of their labels.

That is the whole rule. Three moving parts:

- **distance** — to find who is "closest" (Euclidean distance, from notebook 02);
- **k** — how many neighbours get a vote;
- **the vote** — the most common label among those k neighbours becomes the prediction.

Notice what is *missing*: there is no equation to solve, no curve to fit, nothing the algorithm
learns by fitting. The training data **is** the model. To predict, k-NN looks back at the examples it
has already seen. We will write exactly that, by hand, next.

In [ ]:
# A new point we want to classify (it is not in the training set).
q = np.array([-0.23, 0.75])

# Step 1 - Euclidean distance from q to every training point (notebook 02).
dist = np.sqrt(((X_train - q) ** 2).sum(axis=1))

# Step 2 - the indices of the 5 nearest training points.
nn_idx = np.argsort(dist)[:5]

# Step 3 - look at those 5 neighbours: their distances and their labels.
nn_labels = y_train[nn_idx]
print("5 nearest - distances:", np.round(dist[nn_idx], 3))
print("5 nearest - labels   :", nn_labels)

# Step 4 - the majority vote.
counts = np.bincount(nn_labels, minlength=2)
print(f"votes -> class 0: {counts[0]}   class 1: {counts[1]}")
print("majority vote (k = 5) -> class", counts.argmax())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# All training points, faded, for context.
for cls in (0, 1):
    mask = y_train == cls
    ax.scatter(
        X_train[mask, 0],
        X_train[mask, 1],
        color=CLASS_CYCLE[cls],
        edgecolor=COLORS["text"],
        linewidth=0.4,
        s=40,
        alpha=0.55,
        label=f"class {cls}",
    )

# A dashed line from the query to each of its 5 nearest neighbours.
for j in nn_idx:
    ax.plot(
        [q[0], X_train[j, 0]],
        [q[1], X_train[j, 1]],
        color=COLORS["muted"],
        linewidth=1.0,
        linestyle="--",
        zorder=1,
    )

# Ring the 5 nearest neighbours.
ax.scatter(
    X_train[nn_idx, 0],
    X_train[nn_idx, 1],
    s=240,
    facecolor="none",
    edgecolor=COLORS["highlight"],
    linewidth=2.0,
    zorder=3,
    label="5 nearest",
)

# The query point itself.
ax.scatter(
    q[0],
    q[1],
    marker="*",
    s=480,
    color=COLORS["highlight"],
    edgecolor=COLORS["text"],
    linewidth=0.8,
    zorder=4,
    label="new point q",
)

ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.set_title("q and its 5 nearest neighbours - vote 4 vs 1 -> class 0")
ax.legend(loc="upper right")
plt.show()

**Read the figure.** The star is the new point `q` we want to classify. It sits in the left part of
the plane — and notice that out here, almost every point belongs to class 0: this is the body of the
class-0 crescent, where the class-1 crescent never reaches. The five ringed points are q's nearest
neighbours, with a dashed line drawn to each. Their labels are [1, 0, 0, 0, 0] — four class-0 and
one class-1 — so the vote is 4 to 1, and k-NN predicts **class 0**. The single class-1 ring is the
closest point of all, but it is one stray in a class-0 neighbourhood. That whole decision —
distance, the k nearest, the vote — is the method, in one picture.

## k is the size of the neighbourhood

We used k = 5. But k is a choice, and changing it changes *who gets to vote*. Look again at q: its
single closest neighbour is that one class-1 point. If we set k = 1, the prediction rests entirely
on that one stray. Is a single nearest point a reliable witness — or could it be noise?

Let us watch the vote as we widen the neighbourhood.

In [ ]:
order = np.argsort(dist)  # all training points, nearest first
for k in (1, 3, 5):
    labels_k = y_train[order[:k]]
    vote_k = np.bincount(labels_k, minlength=2).argmax()
    print(f"k = {k}: neighbour labels {labels_k.tolist()} -> vote class {vote_k}")

**Read the output.** At k = 1, the prediction is class 1 — it trusts the single closest point, which
happens to be that stray class-1 neighbour. Widen the neighbourhood to k = 3 or k = 5 and the
class-0 points that actually fill this region outvote the stray, recovering the right label, class
0. This is our first glimpse of a tension we will study closely: **too small a k chases noise**,
following whichever single point happens to be nearest. Choosing a good k is the whole subject of
notebook 3 — for now, the lesson is that k controls how much company a prediction keeps.

## k-NN is a "lazy" learner

Most methods you will meet later work hard up front: they spend real effort during *training* to
distil the data into a compact model (a handful of weights, a tree, a boundary), and then prediction
is quick. k-NN is the mirror image. Its "training" does almost nothing — it only keeps the data. All
the work is deferred to **predict** time, when, for each new point, it measures the distance to every
stored example and finds the nearest ones.

This style has a name: **lazy** (or **instance-based**) learning. Let us feel what it costs.

In [ ]:
# "Train" a k-NN: keep a reference to the data. That is the whole of it.
def knn_fit(features, labels):
    return features, labels


# Predict one point: distance to every stored point, then the vote of the k nearest.
def knn_predict_one(model, query, k=5):
    features, labels = model
    d = np.sqrt(((features - query) ** 2).sum(axis=1))
    nearest = np.argsort(d)[:k]
    return np.bincount(labels[nearest], minlength=2).argmax()


query = np.array([-0.23, 0.75])
for n in (300, 3000, 30000):
    Xn, yn = make_moons(n_samples=n, noise=0.30, random_state=0)

    t0 = time.perf_counter()
    model = knn_fit(Xn, yn)
    fit_us = (time.perf_counter() - t0) * 1e6

    # Average one prediction over many repeats for a stable measurement.
    repeats = 100
    t0 = time.perf_counter()
    for _ in range(repeats):
        knn_predict_one(model, query)
    predict_ms = (time.perf_counter() - t0) / repeats * 1e3

    print(f"n = {n:>6}:  fit {fit_us:7.2f} us   |   predict one point {predict_ms:7.4f} ms")

**Read the output.** Fitting is essentially free and stays flat as the data grows — k-NN only keeps
a reference to the points, which costs about the same whether there are hundreds or tens of thousands
of them. Predicting is where the cost lives: watch the predict column climb as n goes from 300 to
30 000. It grows in step with the number of stored points — and a little faster still, because each
prediction not only measures the distance to every stored point but then ranks them to find the
nearest.

So k-NN's signature is the opposite of most methods: **cheap to train, expensive to predict, and it
must keep the entire dataset in memory.** (Libraries soften this by building a spatial index — a
KD-tree or ball-tree — during fit, which lets them skip most comparisons; but the cost still lives
at predict, and it still grows with the data.) This honest trade-off is worth remembering as we meet
the "eager" learners in later chapters, which pay up front and then predict in a flash.

## Your turn

A few questions to make the rule yours. Reason them out before reaching for code.

1. A query's three nearest neighbours have labels [1, 0, 1]. What does k = 3 predict? What would
   k = 1 predict for the same query?
2. In our example, k = 1 was fooled while k = 3 and k = 5 were not. In your own words, why can a
   single nearest neighbour mislead a prediction near noisy or overlapping data?
3. Suppose you triple the size of the training set. What happens to the time it takes to label one
   new point, and why? What happens to the time it takes to "fit"?

## What you built

You wrote a working classifier by hand — your first real machine-learning method. Given a new point,
you measured its distance to every training example, took the k nearest, and let them vote. You saw
that **k** sets the size of that neighbourhood, that too small a k can chase noise, and that k-NN is
**lazy**: it stores the data and does its thinking at predict time.

**Vocabulary**

- **k-nearest neighbours (k-NN)** — classify a point by the majority vote of its k closest training
  points.
- **neighbour** — a training point, ranked by its distance to the query.
- **the (majority) vote** — the most common label among the k neighbours; it becomes the prediction.
- **k** — the number of neighbours that vote; the size of the neighbourhood.
- **lazy / instance-based learner** — a method whose "fit" only stores the data; the work happens at
  predict time.
- **predict-time cost** — for k-NN, the effort of labelling a point, which grows with the
  training-set size.

## References

- T. M. Cover, P. E. Hart (1967). *Nearest neighbor pattern classification.* IEEE Transactions on
  Information Theory, 13(1):21–27. DOI:
  [10.1109/TIT.1967.1053964](https://doi.org/10.1109/TIT.1967.1053964) — the method's origin, and
  the result that the 1-NN error rate is asymptotically at most twice the best achievable (Bayes)
  rate.
- E. Fix, J. L. Hodges (1951). *Discriminatory analysis: nonparametric discrimination.* USAF School
  of Aviation Medicine — the root of the idea.
- G. James, D. Witten, T. Hastie, R. Tibshirani (2021). *An Introduction to Statistical Learning*
  (§2.2.3). DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1).

---

*Previous: — (you have finished module 00)* · *Next: 02 — Distance & the scale trap*